# Analysis — LEO Wholesale Intelligence


In [1]:
# ============================================================
# 0.1 Imports, paths, load clean data
# ============================================================
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

CLEAN_DIR     = Path("../data/clean")
ANALYTICS_DIR = Path("../data/analytics")      # results read by the backend
ANALYTICS_DIR.mkdir(parents=True, exist_ok=True)

sales = pd.read_parquet(CLEAN_DIR / "sales.parquet")
cust  = pd.read_parquet(CLEAN_DIR / "customers.parquet")
prod  = pd.read_parquet(CLEAN_DIR / "products.parquet")
loc   = pd.read_parquet(CLEAN_DIR / "locations.parquet")
inv   = pd.read_parquet(CLEAN_DIR / "inventory.parquet")

for name, df in {"sales": sales, "customers": cust, "products": prod,
                 "locations": loc, "inventory": inv}.items():
    print(f"{name:<10} {df.shape}")

# Reference date = the day after the last sale.
# All "days since" are measured from here — NOT from today —
# because the data ends on 2026-09-30.
REF_DATE = sales["Order_Date"].max() + pd.Timedelta(days=1)
print("\nReference date:", REF_DATE.date())

sales      (180000, 21)
customers  (30000, 8)
products   (59, 10)
locations  (5, 9)
inventory  (6372, 17)

Reference date: 2026-10-01


In [2]:
# ============================================================
# A.1 Enrich sales with customer and product attributes (joins)
# ============================================================
# These columns were dropped from sales during cleaning (normalization);
# we bring them back with a merge on the ID — one source of truth.
sales["Customer_ID"] = sales["Customer_ID"].astype(str)   # match the type in customers
sales["Product_ID"]  = sales["Product_ID"].astype(str)

rows_before = len(sales)
sales = sales.merge(cust[["Customer_ID", "Customer_Type", "Governorate"]],
                    on="Customer_ID", how="left")
sales = sales.merge(prod[["Product_ID", "Product_Family", "Audience", "Gender", "Season"]],
                    on="Product_ID", how="left")

# A left merge must NEVER add or remove rows (keys are unique in customers/products)
print("Rows before:", rows_before, "| after:", len(sales), "->",
      "OK" if rows_before == len(sales) else "PROBLEM: duplicated keys!")
print("Missing Customer_Type after merge:", sales["Customer_Type"].isna().sum())

Rows before: 180000 | after: 180000 -> OK
Missing Customer_Type after merge: 0


In [3]:
# ============================================================
# A.2 "Hidden traders": Retail-registered customers buying at the Wholesale Desk
# ============================================================
hidden_lines = sales[(sales["Channel"] == "Wholesale") & (sales["Customer_Type"] == "Retail")]

hidden = hidden_lines.groupby("Customer_ID").agg(
    Wholesale_Orders  = ("Invoice_ID", "count"),
    Wholesale_Units   = ("Quantity", "sum"),
    Wholesale_Revenue = ("Net_Sales_EGP", "sum"),
    Last_Wholesale    = ("Order_Date", "max"),
).sort_values("Wholesale_Revenue", ascending=False)

print(f"Hidden traders (Retail accounts buying wholesale): {len(hidden):,}")
print(f"Their wholesale revenue: {hidden['Wholesale_Revenue'].sum():,.0f} EGP")
print("\nOrders per hidden trader:")
print(hidden["Wholesale_Orders"].describe().round(1))

# Join opt-in status (can the bot contact them?)
hidden = hidden.join(cust.set_index("Customer_ID")[["Governorate", "Is_Opted_In"]])
hidden.to_parquet(ANALYTICS_DIR / "hidden_traders.parquet")
print("\nSaved: hidden_traders.parquet")
hidden.head(10)

Hidden traders (Retail accounts buying wholesale): 1,155
Their wholesale revenue: 15,508,260 EGP

Orders per hidden trader:
count    1155.0
mean        1.7
std         1.2
min         1.0
25%         1.0
50%         1.0
75%         2.0
max        10.0
Name: Wholesale_Orders, dtype: float64

Saved: hidden_traders.parquet


,Wholesale_Orders,Wholesale_Units,Wholesale_Revenue,Last_Wholesale,Governorate,Is_Opted_In
Customer_ID,,,,,,
C11487,10,306,93009.48,2023-12-19,Monufia,True
C12388,6,294,81144.24,2024-01-28,Giza,True
C02023,8,306,79167.36,2023-12-08,Alexandria,False
C07560,5,234,70401.48,2023-11-04,Alexandria,False
C09314,8,270,69484.32,2024-02-24,Kafr El Sheikh,True
C09713,6,252,65818.80,2024-01-11,Alexandria,True
C15103,7,246,62921.34,2023-12-09,Cairo,True
C14094,7,252,58614.48,2023-10-20,Alexandria,True
C23857,7,210,53937.36,2023-12-09,Alexandria,False


In [4]:
# ============================================================
# A.3 One row per wholesale trader: purchase behaviour + profitability
# ============================================================
ws = sales[sales["Customer_Type"] == "Wholesale"]   # ALL purchases by wholesale accounts

traders = ws.groupby("Customer_ID").agg(
    First_Order   = ("Order_Date", "min"),
    Last_Order    = ("Order_Date", "max"),
    Orders        = ("Invoice_ID", "count"),       # 1 line = 1 invoice (verified in cleaning)
    Units         = ("Quantity", "sum"),
    Revenue       = ("Net_Sales_EGP", "sum"),
    Gross_Sales   = ("Gross_Sales_EGP", "sum"),
    Profit        = ("Gross_Profit_EGP", "sum"),
    Avg_Discount  = ("Discount_Pct", "mean"),
    Products      = ("Product_ID", "nunique"),
    Returns       = ("Is_Returned", "sum"),
    Size_Returns  = ("Return_Reason", lambda r: (r == "Size / fit").sum()),
)

traders["Recency_Days"]    = (REF_DATE - traders["Last_Order"]).dt.days
traders["Tenure_Days"]     = (REF_DATE - traders["First_Order"]).dt.days
traders["Margin"]          = traders["Profit"] / traders["Revenue"]
traders["Avg_Order_Value"] = traders["Revenue"] / traders["Orders"]
traders["Return_Rate"]     = traders["Returns"] / traders["Orders"]

print(f"Wholesale traders with at least one order: {len(traders):,}")
traders.describe().T.round(2)

Wholesale traders with at least one order: 1,599


,count,mean,min,25%,50%,75%,max,std
First_Order,1599,2025-04-07 18:30:23.639774,2023-10-01 00:00:00,2024-07-13 12:00:00,2025-04-07 00:00:00,2026-01-09 12:00:00,2026-09-30 00:00:00,NaN
Last_Order,1599,2026-08-26 21:39:30.731707,2025-12-11 00:00:00,2026-08-12 00:00:00,2026-09-04 00:00:00,2026-09-19 12:00:00,2026-09-30 00:00:00,NaN
Orders,1599.0,28.109443,1.0,9.0,21.0,39.0,120.0,24.12374
Units,1599.0,884.611632,12.0,282.0,678.0,1260.0,3464.0,755.285901
Revenue,1599.0,215116.0298,3166.08,67361.64,161752.32,307902.36,876177.02,185838.493925
Gross_Sales,1599.0,319182.432145,4656.0,100911.0,240462.0,456369.0,1299644.0,275696.712911
Profit,1599.0,25484.723171,-447.97,7853.47,19115.55,37200.145,115155.14,22517.677921
Avg_Discount,1599.0,0.325665,0.28,0.32,0.325556,0.330833,0.38,0.012596
Products,1599.0,18.860538,1.0,8.0,18.0,28.0,52.0,12.294109
Returns,1599.0,0.507192,0.0,0.0,0.0,1.0,5.0,0.832561


In [5]:
# ============================================================
# A.4 RFM scores (1 = worst, 5 = best) using quintiles
# ============================================================
def quintile_score(series, higher_is_better=True):
    """Split traders into 5 equal-size groups and score them 1–5."""
    # rank(method="first") breaks ties so qcut can always make 5 equal groups
    ranks = series.rank(method="first")
    score = pd.qcut(ranks, q=5, labels=[1, 2, 3, 4, 5]).astype(int)
    return score if higher_is_better else 6 - score

traders["R"] = quintile_score(traders["Recency_Days"], higher_is_better=False)  # fewer days = better
traders["F"] = quintile_score(traders["Orders"])
traders["M"] = quintile_score(traders["Revenue"])
traders["P"] = quintile_score(traders["Profit"])     # our extra dimension: profitability

traders["RFM"] = traders["R"].astype(str) + traders["F"].astype(str) + traders["M"].astype(str)

print("Average behaviour per R score (check the direction is correct):")
print(traders.groupby("R")[["Recency_Days", "Orders", "Revenue"]].mean().round(0))

Average behaviour per R score (check the direction is correct):
   Recency_Days  Orders   Revenue
R                                
1          84.0    27.0  205554.0
2          45.0    27.0  205343.0
3          27.0    29.0  225966.0
4          14.0    29.0  219502.0
5           5.0    29.0  219248.0


In [8]:
# ============================================================
# A.5 Segments from RFM scores (business-readable labels)
# ============================================================
R, F, M = traders["R"], traders["F"], traders["M"]

conditions = [
    (R >= 4) & (F >= 4) & (M >= 4),          # buy recently, often, and a lot
    (R >= 3) & (F >= 4),                      # buy often, still active
    (R <= 2) & (F >= 3),                      # used to buy often, stopped recently
    (R >= 4) & (F <= 2),                      # recent but few orders (new / growing)
    (R <= 2) & (F <= 2),                      # rarely bought and not recently
]
choices = ["Champions", "Loyal", "At Risk", "Promising", "Hibernating"]

traders["Segment"] = np.select(conditions, choices, default="Needs Attention")

# Flag only the real outliers: big buyers in the bottom 10% of margin
margin_p10 = traders["Margin"].quantile(0.10)
traders["Low_Margin_Big_Buyer"] = (traders["M"] >= 4) & (traders["Margin"] < margin_p10)
print(f"10th percentile margin: {margin_p10:.1%}")
print(f"Big buyers with below-median margin: {traders['Low_Margin_Big_Buyer'].sum()}")
print("\nTraders per segment:")
print(traders["Segment"].value_counts())

10th percentile margin: 8.6%
Big buyers with below-median margin: 9

Traders per segment:
Segment
At Risk            371
Needs Attention    320
Hibernating        269
Champions          247
Promising          245
Loyal              147
Name: count, dtype: int64


In [9]:
# ============================================================
# A.6 Segment summary
# ============================================================
seg = traders.groupby("Segment").agg(
    Traders      = ("Orders", "count"),
    Revenue      = ("Revenue", "sum"),
    Profit       = ("Profit", "sum"),
    Avg_Recency  = ("Recency_Days", "mean"),
    Avg_Orders   = ("Orders", "mean"),
    Avg_Discount = ("Avg_Discount", "mean"),
)
seg["Margin"]        = seg["Profit"] / seg["Revenue"]
seg["Revenue_Share"] = seg["Revenue"] / seg["Revenue"].sum()
seg = seg.sort_values("Revenue", ascending=False)

pd.options.display.float_format = "{:,.2f}".format
display(seg)
pd.reset_option("display.float_format")

at_risk = traders[traders["Segment"] == "At Risk"]
print(f"\nAt Risk: {len(at_risk)} traders, "
      f"{at_risk['Revenue'].sum() / traders['Revenue'].sum():.1%} of wholesale revenue")

,Traders,Revenue,Profit,Avg_Recency,Avg_Orders,Avg_Discount,Margin,Revenue_Share
Segment,,,,,,,,
At Risk,371,"116,242,250.79","13,761,429.96",64.39,40.96,0.33,0.12,0.34
Champions,247,"101,966,549.00","12,032,597.23",9.04,53.65,0.33,0.12,0.30
Loyal,147,"57,724,212.98","6,916,047.13",26.55,51.39,0.33,0.12,0.17
Needs Attention,320,"39,126,813.18","4,644,060.69",19.61,16.02,0.33,0.12,0.11
Hibernating,269,"15,244,990.26","1,777,804.90",64.64,7.47,0.33,0.12,0.04
Promising,245,"13,665,715.44","1,618,132.44",9.92,7.39,0.33,0.12,0.04



At Risk: 371 traders, 33.8% of wholesale revenue


In [10]:
# ============================================================
# A.7 Add customer info, include traders who never ordered, save
# ============================================================
info = cust.set_index("Customer_ID")[["Governorate", "Join_Date", "Is_Opted_In"]]
traders = traders.join(info)

# Wholesale accounts with ZERO orders -> keep them (bot target: "first order" offer)
all_ws_ids = cust.loc[cust["Customer_Type"] == "Wholesale", "Customer_ID"]
never = info.loc[all_ws_ids[~all_ws_ids.isin(traders.index)]].copy()
never["Segment"] = "Never Ordered"
never["Orders"] = 0
traders = pd.concat([traders, never])

# Recommended action per segment (shown in the app)
actions = {
    "Champions":       "Keep: priority stock + early access to new collections",
    "Loyal":           "Reward: loyalty pricing on repeat orders",
    "At Risk":         "Call today: personal offer to win back",
    "Promising":       "Grow: suggest best-sellers to increase order size",
    "Hibernating":     "Reactivate: clearance-stock offer",
    "Needs Attention": "Monitor: check in this month",
    "Never Ordered":   "Activate: first-order offer",
}
traders["Action"] = traders["Segment"].map(actions)
# Contact route: bot can message opted-in traders; others need a sales call
traders["Contact_By"] = np.where(traders["Is_Opted_In"].fillna(False), "Bot (WhatsApp)", "Sales call")

traders.index.name = "Customer_ID"
traders = traders.reset_index()
traders.to_parquet(ANALYTICS_DIR / "traders.parquet", index=False)

print(f"Saved traders.parquet: {len(traders):,} traders")
print("\nSegment x contact route:")
print(pd.crosstab(traders["Segment"], traders["Contact_By"], margins=True))

Saved traders.parquet: 1,651 traders

Segment x contact route:
Contact_By       Bot (WhatsApp)  Sales call   All
Segment                                          
At Risk                     250         121   371
Champions                   156          91   247
Hibernating                 177          92   269
Loyal                        94          53   147
Needs Attention             215         105   320
Never Ordered                33          19    52
Promising                   150          95   245
All                        1075         576  1651


###  A. Traders — Results
| Item | Result |
|---|---|
| Merge check | 180,000 rows before and after — no duplicated keys |
| Traders with ≥1 order | 1,599 (+ 52 never ordered = 1,651) |
| Recency | median 27 days; every trader ordered since 2025-12-11 → RFM recency is relative |
| Discount per trader | 28%–38%, std 1.3pp → **flat ~33% for everyone** |
| Margin per trader | median 11.8%, std 3pp → profit tracks revenue at trader level |
|| Low-margin big buyers | M ≥ 4 and margin < 10th percentile (8.6%) → 9 traders |

**Segments**
| Segment | Traders | Revenue share | Avg recency (days) | Avg orders |
|---|---|---|---|---|
| At Risk | 371 | 34% | 64 | 41 |
| Champions | 247 | 30% | 9 | 54 |
| Loyal | 147 | 17% | 27 | 51 |
| Needs Attention | 320 | 11% | 20 | 16 |
| Hibernating | 269 | 4% | 65 | 7 |
| Promising | 245 | 4% | 10 | 7 |

**Findings**
- **371 At-Risk traders = 116M EGP (34% of wholesale revenue)**: frequent buyers (~41 orders) silent for ~2 months
- **Flat discount (~33%) regardless of trader value** → opportunity for tiered, value-based wholesale pricing
- 1,155 Retail accounts bought wholesale (15.5M EGP), all last active Oct 2023–Feb 2024, median 1 order → reactivation list, not current hidden traders
- RFM ignores each trader's own buying rhythm → addressed by the churn model (section B)
- Outputs: `traders.parquet`, `hidden_traders.parquet`

---
# B. Churn model — "Who will stop ordering in the next 90 days?"
**Setup (time-based, no leakage):** features from data BEFORE the cutoff → label = no order in the 90 days AFTER.
Train/test on past data, then score every trader as of the reference date.

In [11]:
# ============================================================
# B.1 Feature builder — uses ONLY data before `as_of` (prevents leakage)
# ============================================================
HORIZON = 90   # days ahead we predict

ws = sales[sales["Customer_Type"] == "Wholesale"].copy()   # wholesale accounts' purchases

def build_features(df, as_of):
    """One row per trader, computed only from orders strictly before `as_of`."""
    hist = df[df["Order_Date"] < as_of]
    f = hist.groupby("Customer_ID").agg(
        Orders       = ("Invoice_ID", "count"),
        Revenue      = ("Net_Sales_EGP", "sum"),
        Profit       = ("Gross_Profit_EGP", "sum"),
        Avg_Discount = ("Discount_Pct", "mean"),
        Products     = ("Product_ID", "nunique"),
        Returns      = ("Is_Returned", "sum"),
        First        = ("Order_Date", "min"),
        Last         = ("Order_Date", "max"),
    )
    f["Recency"]     = (as_of - f["Last"]).dt.days
    f["Tenure"]      = (as_of - f["First"]).dt.days
    # Each trader's own rhythm: average days between orders
    f["Avg_Gap"]       = f["Tenure"] / f["Orders"]
    # How "late" they are compared to their own rhythm (2 = twice as long as usual)
    f["Overdue_Ratio"] = f["Recency"] / f["Avg_Gap"].clip(lower=1)
    f["Margin"]        = f["Profit"] / f["Revenue"]
    f["Return_Rate"]   = f["Returns"] / f["Orders"]

    # Recent trend: last 90 days vs. the 90 days before that
    d90  = as_of - pd.Timedelta(days=90)
    d180 = as_of - pd.Timedelta(days=180)
    last90 = hist[hist["Order_Date"] >= d90].groupby("Customer_ID")["Net_Sales_EGP"].agg(["count", "sum"])
    prev90 = hist[(hist["Order_Date"] >= d180) & (hist["Order_Date"] < d90)] \
                 .groupby("Customer_ID")["Net_Sales_EGP"].agg(["count", "sum"])
    f["Orders_Last90"] = last90["count"].reindex(f.index, fill_value=0)
    f["Orders_Prev90"] = prev90["count"].reindex(f.index, fill_value=0)
    f["Revenue_Trend"] = (last90["sum"].reindex(f.index, fill_value=0) + 1) / \
                         (prev90["sum"].reindex(f.index, fill_value=0) + 1)
    return f.drop(columns=["First", "Last"])

In [12]:
# ============================================================
# B.2 Training data: features at CUTOFF, label = no order in the next 90 days
# ============================================================
CUTOFF = REF_DATE - pd.Timedelta(days=HORIZON)
print("Cutoff date:", CUTOFF.date(), "| label window:", CUTOFF.date(), "->", (REF_DATE - pd.Timedelta(days=1)).date())

X = build_features(ws, CUTOFF)

# Label: 1 = churned (no order in the 90 days after the cutoff), 0 = ordered again
future = ws[(ws["Order_Date"] >= CUTOFF) & (ws["Order_Date"] < REF_DATE)]
buyers_after = set(future["Customer_ID"])
y = (~X.index.isin(buyers_after)).astype(int)
y = pd.Series(y, index=X.index, name="Churned")

print(f"Traders in training data: {len(X):,}")
print(f"Churned: {y.sum():,}  ({y.mean():.1%})   |   Stayed: {(y == 0).sum():,}")
print("\nAverage features: churned vs. stayed")
print(X.assign(Churned=y).groupby("Churned")[["Recency", "Avg_Gap", "Overdue_Ratio",
                                              "Orders", "Orders_Last90", "Revenue_Trend"]].mean().round(2))

Cutoff date: 2026-07-03 | label window: 2026-07-03 -> 2026-09-30
Traders in training data: 1,463
Churned: 95  (6.5%)   |   Stayed: 1,368

Average features: churned vs. stayed
         Recency  Avg_Gap  Overdue_Ratio  Orders  Orders_Last90  Revenue_Trend
Churned                                                                       
0          28.63    22.37           1.37   28.10           3.11        3246.55
1          29.14    22.73           1.34   23.42           2.89        4548.15


In [13]:
# ============================================================
# B.3 Train and evaluate two models on traders the model has NEVER seen
# ============================================================
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, classification_report

FEATURES = list(X.columns)

# 75% of traders to learn from, 25% held out to test
# stratify=y keeps the same churn % in both parts
X_train, X_test, y_train, y_test = train_test_split(
    X[FEATURES], y, test_size=0.25, stratify=y, random_state=42)

models = {
    # Logistic Regression: simple, explainable; needs features on the same scale
    "Logistic Regression": make_pipeline(
        StandardScaler(),
        LogisticRegression(class_weight="balanced", max_iter=1000)),
    # Random Forest: many decision trees voting; captures non-linear patterns
    "Random Forest": RandomForestClassifier(
        n_estimators=300, min_samples_leaf=5, class_weight="balanced", random_state=42),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)                         # learn
    proba = model.predict_proba(X_test)[:, 1]           # probability of churn
    results[name] = roc_auc_score(y_test, proba)
    print(f"{name:<20} AUC = {results[name]:.3f}")

# Baselines: is ML better than a single simple rule?
print(f"\n{'Rule: Recency only':<20} AUC = {roc_auc_score(y_test, X_test['Recency']):.3f}")
print(f"{'Rule: Overdue_Ratio':<20} AUC = {roc_auc_score(y_test, X_test['Overdue_Ratio']):.3f}")

best_name = max(results, key=results.get)
best_model = models[best_name]
print(f"\nBest model: {best_name}")
print(classification_report(y_test, best_model.predict(X_test), target_names=["Stayed", "Churned"]))

Logistic Regression  AUC = 0.451
Random Forest        AUC = 0.488

Rule: Recency only   AUC = 0.518
Rule: Overdue_Ratio  AUC = 0.530

Best model: Random Forest
              precision    recall  f1-score   support

      Stayed       0.93      0.92      0.93       342
     Churned       0.07      0.08      0.07        24

    accuracy                           0.86       366
   macro avg       0.50      0.50      0.50       366
weighted avg       0.88      0.86      0.87       366



In [14]:
# ============================================================
# B.4 What drives churn? (feature importance of the Random Forest)
# ============================================================
rf = models["Random Forest"]
importance = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=False)
print("Feature importance (Random Forest):")
print((importance * 100).round(1).astype(str) + "%")

Feature importance (Random Forest):
Recency          10.1%
Revenue_Trend     9.4%
Profit            9.0%
Avg_Gap           8.5%
Margin            8.4%
Overdue_Ratio     8.3%
Revenue           8.2%
Avg_Discount      8.2%
Tenure            7.8%
Orders            5.6%
Return_Rate       4.8%
Products          4.4%
Orders_Last90     3.7%
Orders_Prev90     2.6%
Returns           0.9%
dtype: str


In [15]:
# ============================================================
# B.5 Score every trader TODAY (as of REF_DATE) for the next 90 days
# ============================================================
# Retrain the best model on ALL labelled traders (more data = better model)
best_model.fit(X[FEATURES], y)

X_now = build_features(ws, REF_DATE)                       # features as of today
X_now["Churn_Prob"] = best_model.predict_proba(X_now[FEATURES])[:, 1]

# Risk levels by rank: top 20% = High, next 30% = Medium, rest = Low
pct_rank = X_now["Churn_Prob"].rank(pct=True)
X_now["Churn_Risk"] = np.select([pct_rank > 0.80, pct_rank > 0.50], ["High", "Medium"], default="Low")

print(X_now["Churn_Risk"].value_counts())
print("\nAverage churn probability by risk level:")
print(X_now.groupby("Churn_Risk")["Churn_Prob"].mean().round(2))

Churn_Risk
Low       799
Medium    480
High      320
Name: count, dtype: int64

Average churn probability by risk level:
Churn_Risk
High      0.48
Low       0.17
Medium    0.31
Name: Churn_Prob, dtype: float64


In [16]:
# ============================================================
# B.6 Save FACTUAL rhythm columns + a transparent rule (no ML scores)
# ============================================================
# The churn model scored AUC ~0.49 (random): churners and non-churners look the
# same in this data. We do NOT ship its probabilities. Instead we keep the
# factual "rhythm" columns and a simple, explainable rule.
cols = ["Avg_Gap", "Overdue_Ratio", "Orders_Last90", "Orders_Prev90"]
traders = traders.drop(columns=cols + ["Churn_Prob", "Churn_Risk", "Revenue_Trend",
                                       "Overdue_Flag"], errors="ignore")
traders = traders.merge(X_now[cols].reset_index(), on="Customer_ID", how="left")

# Rule: silent for at least TWICE their usual gap between orders
traders["Overdue_Flag"] = traders["Overdue_Ratio"] >= 2

traders.to_parquet(ANALYTICS_DIR / "traders.parquet", index=False)
print("Saved traders.parquet:", traders.shape)

od = traders[traders["Overdue_Flag"]]
print(f"\nOverdue traders (≥ 2x their usual gap): {len(od)}")
print(f"  their 3-year revenue: {od['Revenue'].sum():,.0f} EGP "
      f"({od['Revenue'].sum() / traders['Revenue'].sum():.1%} of wholesale)")
print(f"  reachable by the bot (opted-in): {od['Is_Opted_In'].fillna(False).sum()}")
print("\nOverdue flag x RFM segment:")
print(pd.crosstab(traders["Segment"], traders["Overdue_Flag"], margins=True))

Saved traders.parquet: (1651, 34)

Overdue traders (≥ 2x their usual gap): 427
  their 3-year revenue: 127,851,674 EGP (37.2% of wholesale)
  reachable by the bot (opted-in): 293

Overdue flag x RFM segment:
Overdue_Flag     False  True   All
Segment                           
At Risk             73   298   371
Champions          247     0   247
Hibernating        173    96   269
Loyal              115    32   147
Needs Attention    319     1   320
Never Ordered       52     0    52
Promising          245     0   245
All               1224   427  1651


###  B. Churn Model — Results (tested and NOT used)
| Item | Result |
|---|---|
| Setup | Features before 2026-07-03; label = no order 2026-07-03 → 2026-09-30 |
| Training traders | 1,463 · churned 95 (6.5%) |
| Logistic Regression AUC | 0.451 |
| Random Forest AUC | 0.488 |
| Baseline: Recency only | 0.518 |
| Baseline: Overdue_Ratio | 0.530 |
| Churned vs. stayed (feature means) | Nearly identical (Recency 29.1 vs 28.6, Overdue 1.34 vs 1.37) |

**Conclusion:** churn in this (modeled) data is not predictable from purchase behaviour — all models ≈ random (AUC ~0.5).
**Decision:** do NOT ship ML churn scores. Use transparent facts instead:
`Avg_Gap` (usual days between orders), `Overdue_Ratio` (days silent ÷ usual gap), and the rule
**`Overdue_Flag` = silent ≥ 2× usual gap** → _fill in count and revenue share from B.6_.
"At Risk" is presented as a factual description (frequent buyers silent ~64 days), not a prediction.
The pipeline (`build_features` + time-based split) is ready to retrain on the company's real data.

---
# C. Sizes and demand forecast (30 / 60 / 90 days)
**Size question:** does the size mix differ by audience, season, channel or product?
**Forecast:** monthly demand per product (from inventory `Demand_Qty` = sold + lost sales),
backtested on the last 3 months before forecasting Oct–Dec 2026.

In [17]:
# ============================================================
# C.1 Size mix: does it vary by audience, season, channel, family or product?
# ============================================================
KIDS_SIZES  = ["4", "6", "8", "10", "12"]
WOMEN_SIZES = ["M", "L", "XL", "2XL"]

def size_shares(df, by):
    """Share of UNITS per size inside each group (rows = groups, columns = sizes)."""
    units = df.groupby(by + ["Size"], observed=True)["Quantity"].sum()
    shares = units / units.groupby(level=by).transform("sum")
    return shares.unstack("Size")

def spread_pp(shares):
    """Biggest minus smallest size share in each row, in percentage points."""
    return ((shares.max(axis=1) - shares.min(axis=1)) * 100).round(1)

for aud, sizes in [("Kids", KIDS_SIZES), ("Women", WOMEN_SIZES)]:
    d = sales[sales["Audience"] == aud]
    print(f"\n==================== {aud} ====================")
    overall = size_shares(d.assign(All="All"), ["All"])[sizes]
    print("Overall share of units:\n", (overall * 100).round(1).to_string())
    for dim in ["Channel", "Season", "Product_Family"]:
        sh = size_shares(d, [dim])[sizes]
        print(f"\nBy {dim} — spread (max − min share, pp):")
        print(spread_pp(sh).to_string())
    per_product = spread_pp(size_shares(d, ["Product_ID"])[sizes])
    print(f"\nPer product — spread in pp: median {per_product.median()}, max {per_product.max()}")


==================== Kids ====================
Overall share of units:
 Size     4     6     8    10    12
All                               
All   20.0  20.0  20.1  20.3  19.7

By Channel — spread (max − min share, pp):
Channel
Online       0.5
Retail       0.4
Wholesale    0.7

By Season — spread (max − min share, pp):
Season
Autumn/Winter    1.5
Summer           0.3

By Product_Family — spread (max − min share, pp):
Product_Family
Kids Sets           1.5
Pajamas/Homewear    0.1
Sweatshirts         2.0
T-Shirts            1.5

Per product — spread in pp: median 3.8499999999999996, max 8.7

==================== Women ====================
Overall share of units:
 Size     M     L    XL   2XL
All                         
All   24.5  25.5  25.1  24.9

By Channel — spread (max − min share, pp):
Channel
Online       1.0
Retail       1.2
Wholesale    1.1

By Season — spread (max − min share, pp):
Season
Summer    0.9

By Product_Family — spread (max − min share, pp):
Product_Family
Pajamas

In [18]:
# ============================================================
# C.2 Size/fit returns: which sizes and products have fit problems?
# ============================================================
is_size_ret = sales["Return_Reason"] == "Size / fit"
sales["_size_ret"] = is_size_ret

by_size = sales.groupby(["Audience", "Size"], observed=True).agg(
    Lines        = ("Invoice_ID", "count"),
    Size_Returns = ("_size_ret", "sum"),
)
by_size["Size_Return_Rate_%"] = (by_size["Size_Returns"] / by_size["Lines"] * 100).round(2)
print("Size/fit return rate by size:")
print(by_size.to_string())

by_prod = sales.groupby("Product_ID").agg(
    Lines        = ("Invoice_ID", "count"),
    Size_Returns = ("_size_ret", "sum"),
)
by_prod["Size_Return_Rate_%"] = (by_prod["Size_Returns"] / by_prod["Lines"] * 100).round(2)
overall_rate = is_size_ret.mean() * 100
print(f"\nOverall size/fit return rate: {overall_rate:.2f}%")

# Products with clearly more fit problems (enough sales to trust the rate)
by_prod = by_prod.join(prod.set_index("Product_ID")[["Product_Name", "Audience", "Product_Family"]])
worst = by_prod[by_prod["Lines"] >= 500].sort_values("Size_Return_Rate_%", ascending=False)
print("\nTop 10 products by size/fit return rate (≥ 500 lines):")
print(worst.head(10)[["Product_Name", "Audience", "Lines", "Size_Return_Rate_%"]].to_string())

by_prod.reset_index().to_parquet(ANALYTICS_DIR / "size_returns_by_product.parquet", index=False)
sales = sales.drop(columns="_size_ret")
print("\nSaved: size_returns_by_product.parquet")

Size/fit return rate by size:
               Lines  Size_Returns  Size_Return_Rate_%
Audience Size                                         
Kids     10    30355           530                1.75
         12    29934           476                1.59
         4     30144           487                1.62
         6     29918           506                1.69
         8     30304           525                1.73
Women    2XL    7348           133                1.81
         L      7316           143                1.95
         M      7283           137                1.88
         XL     7398           113                1.53

Overall size/fit return rate: 1.69%

Top 10 products by size/fit return rate (≥ 500 lines):
                                        Product_Name Audience  Lines  Size_Return_Rate_%
Product_ID                                                                              
P054                Gazelle - بيجامة حريمي صيفي 2024    Women   1686                2.55
P058 

In [19]:
# ============================================================
# C.3 Monthly demand per product (all locations), from inventory Demand_Qty
# ============================================================
# Demand_Qty = what customers wanted (sold + lost sales) -> the right thing to forecast
inv["Product_ID"] = inv["Product_ID"].astype(str)
demand = (inv.groupby(["Snapshot_Month", "Product_ID"], observed=True)["Demand_Qty"].sum()
             .unstack("Product_ID")
             .sort_index())
print("Demand matrix (months x products):", demand.shape)
print("Months:", demand.index.min().date(), "->", demand.index.max().date())

# Sanity: inventory units sold vs. sales-file units (should be close)
inv_sold = inv["Sold_Qty"].sum()
sales_units = sales["Quantity"].sum()
print(f"\nUnits sold — inventory: {inv_sold:,.0f} | sales file: {sales_units:,.0f} "
      f"(ratio {inv_sold / sales_units:.2f})")

# Company-wide monthly demand: seasonality at a glance
total = demand.sum(axis=1)
print("\nTotal monthly demand (last 15 months):")
print(total.tail(15).astype(int).to_string())
print("\nAverage demand by calendar month:")
print(total.groupby(total.index.month).mean().round(0).astype(int).to_string())

Demand matrix (months x products): (36, 59)
Months: 2023-10-01 -> 2026-09-01

Units sold — inventory: 1,644,885 | sales file: 1,660,589 (ratio 0.99)

Total monthly demand (last 15 months):
Snapshot_Month
2025-07-01    55348
2025-08-01    54648
2025-09-01    41569
2025-10-01    37453
2025-11-01    42307
2025-12-01    58762
2026-01-01    40138
2026-02-01    35362
2026-03-01    48201
2026-04-01    50135
2026-05-01    57685
2026-06-01    56301
2026-07-01    59539
2026-08-01    54091
2026-09-01    44198

Average demand by calendar month:
Snapshot_Month
1     37857
2     33670
3     44325
4     49284
5     52926
6     52261
7     55699
8     52183
9     41286
10    38560
11    42428
12    53050


In [20]:
# ============================================================
# C.4 Backtest: which simple forecasting method works best on THIS data?
# Hold out the last 3 months, forecast them from earlier data, compare to reality.
# ============================================================
H = 3   # months ahead (≈ 30 / 60 / 90 days)

def future_index(history, h):
    return pd.date_range(history.index[-1] + pd.offsets.MonthBegin(1), periods=h, freq="MS")

def f_moving_avg(history, h):
    """Average of the last 3 months, repeated."""
    return pd.DataFrame([history.iloc[-3:].mean()] * h, index=future_index(history, h))

def f_seasonal_naive(history, h):
    """Same months last year."""
    return pd.DataFrame(history.iloc[-12:-12 + h].values, index=future_index(history, h),
                        columns=history.columns)

def f_seasonal_trend(history, h):
    """Same months last year x each product's growth (last 12 months vs. the 12 before)."""
    growth = (history.iloc[-12:].sum() / history.iloc[-24:-12].sum()).clip(0.5, 2).fillna(1)
    return f_seasonal_naive(history, h) * growth

def wape(forecast, actual):
    """Weighted Absolute Percentage Error: total absolute error / total actual demand."""
    return (forecast - actual).abs().sum().sum() / actual.sum().sum()

train, actual = demand.iloc[:-H], demand.iloc[-H:]
methods = {"3-month moving average": f_moving_avg,
           "Same month last year":   f_seasonal_naive,
           "Last year x growth":     f_seasonal_trend}

scores = {}
for name, fn in methods.items():
    fc = fn(train, H)
    fc.index = actual.index
    scores[name] = wape(fc, actual)
    print(f"{name:<25} WAPE = {scores[name]:.1%}   (accuracy ≈ {1 - scores[name]:.1%})")

best_method = min(scores, key=scores.get)
print(f"\nBest method: {best_method}")

3-month moving average    WAPE = 28.6%   (accuracy ≈ 71.4%)
Same month last year      WAPE = 26.8%   (accuracy ≈ 73.2%)
Last year x growth        WAPE = 27.2%   (accuracy ≈ 72.8%)

Best method: Same month last year


In [21]:
# ============================================================
# C.5 Forecast Oct–Dec 2026 with the best method, then split by size
# ============================================================
fc = methods[best_method](demand, H).clip(lower=0).round()
print("Forecast months:", [d.strftime("%Y-%m") for d in fc.index])

forecast = pd.DataFrame({
    "Forecast_30d": fc.iloc[0],
    "Forecast_60d": fc.iloc[:2].sum(),
    "Forecast_90d": fc.iloc[:3].sum(),
})
forecast.index.name = "Product_ID"
forecast = forecast.join(prod.set_index("Product_ID")[["Product_Name", "Audience", "Product_Family", "Season"]])
print(f"\nTotal forecast — 30d: {forecast['Forecast_30d'].sum():,.0f} | "
      f"60d: {forecast['Forecast_60d'].sum():,.0f} | 90d: {forecast['Forecast_90d'].sum():,.0f} units")

# Size split: each product's size shares over the LAST 12 MONTHS of sales
last12 = sales[sales["Order_Date"] >= REF_DATE - pd.DateOffset(months=12)]
su = last12.groupby(["Product_ID", "Size"], observed=True)["Quantity"].sum()
size_share = (su / su.groupby(level="Product_ID").transform("sum")).rename("Size_Share").reset_index()

by_size = size_share.merge(forecast[["Forecast_90d"]].reset_index(), on="Product_ID")
by_size["Forecast_90d_Size"] = (by_size["Forecast_90d"] * by_size["Size_Share"]).round()

forecast.reset_index().to_parquet(ANALYTICS_DIR / "forecast.parquet", index=False)
by_size.to_parquet(ANALYTICS_DIR / "forecast_by_size.parquet", index=False)
print("Saved: forecast.parquet + forecast_by_size.parquet")
forecast.sort_values("Forecast_90d", ascending=False).head(10)

Forecast months: ['2026-10', '2026-11', '2026-12']

Total forecast — 30d: 37,453 | 60d: 79,760 | 90d: 138,522 units
Saved: forecast.parquet + forecast_by_size.parquet


,Forecast_30d,Forecast_60d,Forecast_90d,Product_Name,Audience,Product_Family,Season
Product_ID,,,,,,,
P037,3084,7060,11155,Girls' sweatshirt-Flower,Kids,Sweatshirts,Autumn/Winter
P033,2283,4478,7935,Boys' sweatshirt-LEO,Kids,Sweatshirts,Autumn/Winter
P038,1640,3844,7597,Boys' Autumn Pajamas-Mickey Mouse,Kids,Pajamas/Homewear,Autumn/Winter
P032,1750,3890,7140,Boys' sweatshirt-blue,Kids,Sweatshirts,Autumn/Winter
P039,1954,4001,7058,Boys' Autumn Pajamas-Spider Man,Kids,Pajamas/Homewear,Autumn/Winter
P043,1805,3682,6753,Girls' autumn pajamas-Snow White,Kids,Pajamas/Homewear,Autumn/Winter
P045,1507,3588,6515,Girls' autumn pajamas-Stitch,Kids,Pajamas/Homewear,Autumn/Winter
P044,1618,3596,6103,Girls' autumn pajamas-Ice Cream,Kids,Pajamas/Homewear,Autumn/Winter
P035,882,1991,3711,Girls' sweatshirt-Fuchsia,Kids,Sweatshirts,Autumn/Winter


In [22]:
# ============================================================
# C.6 Production need vs. current stock (all locations) + overstock list
# ============================================================
FILL_RATE = 0.91          # factory delivers ~91% of planned qty (production section)
MAX_MONTH = 251_053       # highest monthly output actually achieved (practical ceiling)

last_month = inv["Snapshot_Month"].max()
stock_now = inv[inv["Snapshot_Month"] == last_month].groupby("Product_ID", observed=True)["Closing_Stock"].sum()

plan = forecast[["Product_Name", "Audience", "Forecast_90d"]].copy()
plan["Stock_Now"] = stock_now.reindex(plan.index).fillna(0)
plan["Net_Need_90d"] = (plan["Forecast_90d"] - plan["Stock_Now"]).clip(lower=0)
# Plan extra because the factory produces ~91% of what is planned
plan["Plan_To_Produce"] = (plan["Net_Need_90d"] / FILL_RATE).round()
# Cover: how many days current stock lasts at the forecast pace
plan["Days_Cover_Forecast"] = (plan["Stock_Now"] / (plan["Forecast_90d"] / 90)).round(0)

total_plan = plan["Plan_To_Produce"].sum()
print(f"Stock now ({last_month.date()}): {plan['Stock_Now'].sum():,.0f} units")
print(f"Forecast 90d demand:            {plan['Forecast_90d'].sum():,.0f} units")
print(f"Plan to produce (90d):          {total_plan:,.0f} units "
      f"≈ {total_plan / 3:,.0f}/month ({total_plan / 3 / MAX_MONTH:.0%} of best month ever)")

over = plan[plan["Days_Cover_Forecast"] > 180].sort_values("Stock_Now", ascending=False)
print(f"\nOverstocked products (> 180 days of cover at forecast pace): {len(over)}")
print(f"  units tied up: {over['Stock_Now'].sum():,.0f}")
print(over.head(10)[["Product_Name", "Stock_Now", "Forecast_90d", "Days_Cover_Forecast"]].to_string())

plan.reset_index().to_parquet(ANALYTICS_DIR / "production_plan.parquet", index=False)
print("\nSaved: production_plan.parquet")

Stock now (2026-09-01): 77,867 units
Forecast 90d demand:            138,522 units
Plan to produce (90d):          105,356 units ≈ 35,119/month (14% of best month ever)

Overstocked products (> 180 days of cover at forecast pace): 4
  units tied up: 40,162
                                  Product_Name  Stock_Now  Forecast_90d  Days_Cover_Forecast
Product_ID                                                                                  
P023               Girls' summer pajamas-Dumbo      10343           412               2259.0
P019            Boys' summer pajamas- Seeds OF      10208           713               1289.0
P041        Boys' Autumn Pajamas-Jurassic Park       9954          3666                244.0
P054          Gazelle - بيجامة حريمي صيفي 2024       9657           742               1171.0

Saved: production_plan.parquet


###  C. Sizes and Forecast — Results
| Item | Result |
|---|---|
| Size mix (Kids) | 19.7%–20.3% per size; spread by channel 0.4–0.7pp, season 0.3–1.5pp, family 0.1–2.0pp |
| Size mix (Women) | 24.5%–25.5% per size; spread ≤ 1.2pp |
| Per-product size spread | median ~3.7pp, max 8.7pp (likely noise) |
| **Conclusion — sizes** | Demand is evenly split across sizes everywhere → no "make more of size X" insight; per-product shares still used to split the forecast |
| Size/fit return rate | 1.69% overall; 1.53%–1.95% across all sizes → not one problem size; it's fit guidance in general |
| Worst products (size/fit) | P054 Gazelle 2.55%, P058 OHANA 2.28%, P019 Seeds 2.25% |
| Inventory vs. sales units | ratio 0.99 ✓ |
| Seasonality | Peak Jul (55.7k), May–Aug > 52k; low Feb (33.7k); Dec spike (53k) |
| Backtest WAPE (Jul–Sep 2026) | Moving avg _fill in_ · Same month LY _fill in_ · LY × growth _fill in_ |
| Best method | Same month last year (seasonality dominates) |
| Forecast Oct–Dec 2026 | 30d 37,453 · 60d 79,760 · 90d 138,522 units (winter sweatshirts/pajamas lead) |
| Production plan 90d (÷ 91% fill rate) | 105,356 units ≈ 35k/month → well within capacity |
| **Overstock** | **4 products hold 40,162 units = 52% of all stock** (P023 Dumbo 2,259 days, P019 Seeds 1,289, P054 Gazelle 1,171, P041 Jurassic Park 244) |

**Links:** two overstocked products (P054, P019) are also top size/fit return products.
**Caveat:** production-order volumes (~174k/month) don't reconcile with demand (~46k/month) in the modeled data → production file used for lead times and bottlenecks only.

---
# D. Stock status + bot catalog
**Stock rules (business thresholds — editable):** Out of stock = 0 · Reorder = at/below reorder point ·
Low = < 30 days cover · Healthy = 30–120 days · Overstock = > 120 days.
**Bot catalog:** one row per product with wholesale price, Wholesale Desk stock, estimated stock per size,
clearance offer, size-fit warning and restock lead time.

In [23]:
# ============================================================
# D.1 Latest stock status per product x location (rule-based)
# ============================================================
inv["Location_ID"] = inv["Location_ID"].astype(str)
latest = inv[inv["Snapshot_Month"] == inv["Snapshot_Month"].max()].copy()

# Business thresholds (days of cover) — explicit and easy to change
LOW_DAYS, OVER_DAYS = 30, 120

latest["Stock_Status"] = np.select(
    [latest["Closing_Stock"] == 0,
     latest["Closing_Stock"] <= latest["Reorder_Point_Units"],
     latest["Days_of_Cover"] < LOW_DAYS,
     latest["Days_of_Cover"] > OVER_DAYS],
    ["Out of stock", "Reorder now", "Low", "Overstock"],
    default="Healthy")

latest = latest.merge(loc[["Location_ID", "Location_Short"]], on="Location_ID", how="left")
print(f"Snapshot: {latest['Snapshot_Month'].max().date()}")
print(pd.crosstab(latest["Location_Short"], latest["Stock_Status"], margins=True))

cols = ["Location_ID", "Location_Short", "Product_ID", "Closing_Stock", "Avg_Daily_Sales_90D",
        "Days_of_Cover", "Reorder_Point_Units", "Supplier_Lead_Time_Days", "Stock_Status"]
latest[cols].to_parquet(ANALYTICS_DIR / "stock_status.parquet", index=False)
print("\nSaved: stock_status.parquet")

Snapshot: 2026-09-01
Stock_Status      Healthy  Low  Out of stock  Overstock  Reorder now  All
Location_Short                                                           
Online Store           26    2             1         29            1   59
Sidi Bishr Store       28    0             0         29            2   59
Wholesale Desk          4   13             6          6           30   59
All                    58   15             7         64           33  177

Saved: stock_status.parquet


In [24]:
# ============================================================
# D.2 Wholesale price per product (actual prices paid in the last 12 months)
# ============================================================
ws12 = sales[(sales["Channel"] == "Wholesale") &
             (sales["Order_Date"] >= REF_DATE - pd.DateOffset(months=12))].copy()
ws12["Unit_Net"]  = ws12["Net_Sales_EGP"] / ws12["Quantity"]
ws12["Unit_COGS"] = ws12["Estimated_COGS_EGP"] / ws12["Quantity"]

price = ws12.groupby("Product_ID").agg(
    Wholesale_Price = ("Unit_Net", "median"),
    Unit_Cost       = ("Unit_COGS", "median"),
    Avg_Discount    = ("Discount_Pct", "mean"),
).round(2)

# Products with no wholesale sale in 12 months -> fall back to catalog price x (1 - typical discount)
typical_disc = ws12["Discount_Pct"].median()
cat = prod.set_index("Product_ID")["Public_Retail_Price_EGP"]
price = price.reindex(cat.index)
price["Wholesale_Price"] = price["Wholesale_Price"].fillna((cat * (1 - typical_disc)).round(2))
print(f"Typical wholesale discount: {typical_disc:.1%}")
print(f"Products using the fallback price: {price['Unit_Cost'].isna().sum()}")
price.head()

Typical wholesale discount: 32.0%
Products using the fallback price: 0


,Wholesale_Price,Unit_Cost,Avg_Discount
Product_ID,,,
P001,236.64,200.02,0.33
P002,236.64,202.48,0.32
P003,236.64,215.92,0.33
P004,236.64,217.73,0.32
P005,236.64,219.20,0.32


In [26]:
# ============================================================
# D.3 Bot catalog: everything the assistant needs, one row per product
# ============================================================
WHOLESALE_LOC = "LOC003"
LEAD_TIME_DAYS = 16                  # median actual production lead time
CLEARANCE_EXTRA_DISC = 0.10          # extra 10% off for overstock clearance
MIN_MARGIN = 0.05                    # never offer below cost + 5%

# Columns we want from the catalog — keep only those that exist
# (Brand was dropped in notebook 01 because it had a single value)
wanted = ["Product_Name", "Brand", "Product_Family", "Gender", "Audience",
          "Season", "Season_Year", "Sizes_List", "Public_Retail_Price_EGP"]
available = [c for c in wanted if c in prod.columns]
missing = [c for c in wanted if c not in prod.columns]
if missing:
    print("Not in catalog (skipped):", missing)

bot = prod.set_index("Product_ID")[available].copy()
bot = bot.join(price)

# Stock at the Wholesale Desk + company-wide
ws_stock = latest[latest["Location_ID"] == WHOLESALE_LOC].set_index("Product_ID")
bot["Stock_Wholesale_Desk"] = ws_stock["Closing_Stock"].reindex(bot.index).fillna(0).astype(int)
bot["Stock_Status_Desk"]    = ws_stock["Stock_Status"].reindex(bot.index).fillna("Unknown")
bot["Stock_All_Locations"]  = plan["Stock_Now"].reindex(bot.index).fillna(0).astype(int)
bot["Days_Cover_Forecast"]  = plan["Days_Cover_Forecast"].reindex(bot.index)

# Clearance offer for overstock — price protected by a minimum margin
bot["Clearance"] = bot["Days_Cover_Forecast"] > 180
floor_price = (bot["Unit_Cost"] * (1 + MIN_MARGIN)).round(2)
offer = (bot["Wholesale_Price"] * (1 - CLEARANCE_EXTRA_DISC)).round(2)
bot["Clearance_Price"] = np.where(bot["Clearance"], np.maximum(offer, floor_price), np.nan)

# Size-fit warning: products clearly above the average size/fit return rate
sr = pd.read_parquet(ANALYTICS_DIR / "size_returns_by_product.parquet").set_index("Product_ID")
bot["Size_Return_Rate_%"] = sr["Size_Return_Rate_%"].reindex(bot.index)
bot["Size_Fit_Warning"] = bot["Size_Return_Rate_%"] >= 2.2

bot["Restock_Lead_Days"] = LEAD_TIME_DAYS
bot["Sizes_List"] = bot["Sizes_List"].apply(list)        # numpy array -> plain list

bot = bot.reset_index()
bot.to_parquet(ANALYTICS_DIR / "bot_catalog.parquet", index=False)
print(f"Saved bot_catalog.parquet: {bot.shape}")
print(f"Clearance offers: {bot['Clearance'].sum()} | size-fit warnings: {bot['Size_Fit_Warning'].sum()}")
bot.loc[bot["Clearance"], ["Product_ID", "Product_Name", "Stock_All_Locations",
                           "Wholesale_Price", "Unit_Cost", "Clearance_Price"]]

Not in catalog (skipped): ['Brand']
Saved bot_catalog.parquet: (59, 21)
Clearance offers: 4 | size-fit warnings: 5


,Product_ID,Product_Name,Stock_All_Locations,Wholesale_Price,Unit_Cost,Clearance_Price
18,P019,Boys' summer pajamas- Seeds OF,10208,203.32,155.11,182.99
22,P023,Girls' summer pajamas-Dumbo,10343,209.30,175.24,188.37
40,P041,Boys' Autumn Pajamas-Jurassic Park,9954,196.52,157.23,176.87
53,P054,Gazelle - بيجامة حريمي صيفي 2024,9657,263.84,216.95,237.46


In [27]:
# ============================================================
# D.4 Estimated stock per size at the Wholesale Desk
# (inventory is tracked per product, not per size -> estimate with recent size shares)
# ============================================================
est = size_share.merge(bot[["Product_ID", "Stock_Wholesale_Desk"]], on="Product_ID")
est["Est_Stock_Size"] = (est["Stock_Wholesale_Desk"] * est["Size_Share"]).round().astype(int)
est = est[["Product_ID", "Size", "Size_Share", "Est_Stock_Size"]]
est["Size"] = est["Size"].astype(str)
est.to_parquet(ANALYTICS_DIR / "stock_by_size_estimate.parquet", index=False)
print("Saved stock_by_size_estimate.parquet:", est.shape)
est.head(10)

Saved stock_by_size_estimate.parquet: (286, 4)


,Product_ID,Size,Size_Share,Est_Stock_Size
0,P001,10,0.191775,95
1,P001,12,0.201144,99
2,P001,4,0.196687,97
3,P001,6,0.207811,103
4,P001,8,0.202583,100
5,P002,10,0.204203,0
6,P002,12,0.192820,0
7,P002,4,0.185621,0
8,P002,6,0.225606,0
9,P002,8,0.191750,0


In [28]:
# ============================================================
# E.1 KPIs for the Overview page (+ monthly series for charts)
# ============================================================
import json

ws_all = sales[sales["Channel"] == "Wholesale"]
tr = traders[traders["Orders"] > 0]
active_90 = (tr["Recency_Days"] <= 90).sum()
at_risk = tr[tr["Segment"] == "At Risk"]
gross_ws = ws_all["Gross_Sales_EGP"].sum()

kpis = {
    "as_of": str(REF_DATE.date()),
    "revenue_total": float(sales["Net_Sales_EGP"].sum()),
    "profit_total": float(sales["Gross_Profit_EGP"].sum()),
    "margin_total": float(sales["Gross_Profit_EGP"].sum() / sales["Net_Sales_EGP"].sum()),
    "wholesale_revenue_share": float(ws_all["Net_Sales_EGP"].sum() / sales["Net_Sales_EGP"].sum()),
    "wholesale_profit_share": float(ws_all["Gross_Profit_EGP"].sum() / sales["Gross_Profit_EGP"].sum()),
    "wholesale_margin": float(ws_all["Gross_Profit_EGP"].sum() / ws_all["Net_Sales_EGP"].sum()),
    "wholesale_avg_discount": float(ws_all["Discount_Pct"].mean()),
    "profit_per_discount_point": float(gross_ws * 0.01),       # 1pp less discount = this much more profit
    "traders_total": int(len(traders)),
    "traders_active_90d": int(active_90),
    "traders_never_ordered": int((traders["Orders"] == 0).sum()),
    "at_risk_traders": int(len(at_risk)),
    "at_risk_revenue_share": float(at_risk["Revenue"].sum() / tr["Revenue"].sum()),
    "overdue_traders": int(traders["Overdue_Flag"].fillna(False).sum()),
    "return_rate": float(sales["Is_Returned"].mean()),
    "size_fit_share_of_returns": float((sales["Return_Reason"] == "Size / fit").sum() / sales["Is_Returned"].sum()),
    "stock_units_now": int(plan["Stock_Now"].sum()),
    "overstock_units": int(plan.loc[plan["Days_Cover_Forecast"] > 180, "Stock_Now"].sum()),
    "forecast_30d": int(forecast["Forecast_30d"].sum()),
    "forecast_60d": int(forecast["Forecast_60d"].sum()),
    "forecast_90d": int(forecast["Forecast_90d"].sum()),
}
with open(ANALYTICS_DIR / "kpis.json", "w", encoding="utf-8") as fh:
    json.dump(kpis, fh, indent=2)

# Monthly revenue & profit by channel (for the trend chart)
monthly = (sales.assign(Month=sales["Order_Date"].dt.to_period("M").dt.to_timestamp())
                .groupby(["Month", "Channel"], observed=True)
                .agg(Revenue=("Net_Sales_EGP", "sum"), Profit=("Gross_Profit_EGP", "sum"),
                     Units=("Quantity", "sum"))
                .reset_index())
monthly.to_parquet(ANALYTICS_DIR / "monthly_by_channel.parquet", index=False)

# Segment summary (for the traders page header)
seg_summary = tr.groupby("Segment").agg(Traders=("Customer_ID", "count"),
                                        Revenue=("Revenue", "sum"),
                                        Avg_Recency=("Recency_Days", "mean")).reset_index()
seg_summary["Revenue_Share"] = seg_summary["Revenue"] / seg_summary["Revenue"].sum()
seg_summary.to_parquet(ANALYTICS_DIR / "segment_summary.parquet", index=False)

print(json.dumps(kpis, indent=2))
print("\nSaved: kpis.json, monthly_by_channel.parquet, segment_summary.parquet")

{
  "as_of": "2026-10-01",
  "revenue_total": 421338377.53,
  "profit_total": 64888289.900000006,
  "margin_total": 0.15400517341997846,
  "wholesale_revenue_share": 0.8529038294272467,
  "wholesale_profit_share": 0.6570529512444431,
  "wholesale_margin": 0.11864122332580129,
  "wholesale_avg_discount": 0.32603638155639353,
  "profit_per_discount_point": 5332309.98,
  "traders_total": 1651,
  "traders_active_90d": 1504,
  "traders_never_ordered": 52,
  "at_risk_traders": 371,
  "at_risk_revenue_share": 0.3379424691772138,
  "overdue_traders": 427,
  "return_rate": 0.047461111111111114,
  "size_fit_share_of_returns": 0.35701744117991335,
  "stock_units_now": 77867,
  "overstock_units": 40162,
  "forecast_30d": 37453,
  "forecast_60d": 79760,
  "forecast_90d": 138522
}

Saved: kpis.json, monthly_by_channel.parquet, segment_summary.parquet


In [29]:
# ============================================================
# E.2 Check every output file the backend will read
# ============================================================
for f in sorted(ANALYTICS_DIR.iterdir()):
    if f.suffix == ".parquet":
        df = pd.read_parquet(f)
        print(f"{f.name:<35} {df.shape[0]:>6,} rows x {df.shape[1]:>2} cols")
    else:
        print(f"{f.name:<35} {f.stat().st_size:>6,} bytes")

bot_catalog.parquet                     59 rows x 21 cols
forecast.parquet                        59 rows x  8 cols
forecast_by_size.parquet               286 rows x  5 cols
hidden_traders.parquet               1,155 rows x  6 cols
kpis.json                              800 bytes
monthly_by_channel.parquet             108 rows x  5 cols
production_plan.parquet                 59 rows x  8 cols
segment_summary.parquet                  6 rows x  5 cols
size_returns_by_product.parquet         59 rows x  7 cols
stock_by_size_estimate.parquet         286 rows x  4 cols
stock_status.parquet                   177 rows x  9 cols
traders.parquet                      1,651 rows x 34 cols


### D. Stock Status + Bot Catalog — Results
| Item | Result |
|---|---|
| Stock rules | Out of stock = 0 · Reorder = ≤ reorder point · Low < 30d · Healthy 30–120d · Overstock > 120d |
| **Wholesale Desk (85% of revenue)** | **30 reorder now, 6 out of stock, 13 low, 4 healthy, 6 overstock** |
| Sidi Bishr / Online (15% of revenue) | 29 overstock each, mostly healthy otherwise |
| Typical wholesale discount | 32.0% → kids pajama wholesale price ≈ 236.64 EGP (348 × 0.68) |
| Clearance offers | 4 products (P019, P023, P041, P054), extra 10% off, never below cost + 5% |
| Size-fit warnings (≥ 2.2%) | 5 products |
| Stock per size | Estimated = Wholesale Desk stock × last-12-month size share (bot must say "approximately") |

**Key insight:** stock sits in the wrong place — the Wholesale Desk is short on 49 of 59 products
while the stores hold overstock → **rebalance stores → Wholesale Desk before producing more**.

###  E. KPIs — Results (as of 2026-10-01)
| KPI | Value |
|---|---|
| Revenue / Profit / Margin | 421.3M EGP / 64.9M EGP / 15.4% |
| Wholesale share of revenue / profit | 85.3% / 65.7% |
| Wholesale margin / avg discount | 11.9% / 32.6% |
| **Profit per 1 point of wholesale discount** | **5.33M EGP** |
| Traders: total / active 90d / never ordered | 1,651 / 1,504 / 52 |
| At Risk traders / revenue share | 371 / 33.8% |
| Overdue traders (≥ 2× usual gap) | 427 |
| Return rate / size-fit share of returns | 4.7% / 35.7% |
| Stock now / overstock units | 77,867 / 40,162 (52%) |
| Forecast 30 / 60 / 90 days | 37,453 / 79,760 / 138,522 units |

**Outputs in `data/analytics/` (12 files):** bot_catalog, forecast, forecast_by_size, hidden_traders,
kpis.json, monthly_by_channel, production_plan, segment_summary, size_returns_by_product,
stock_by_size_estimate, stock_status, traders.